<a href="https://colab.research.google.com/github/lachlandachlan450/llm-encoded-chainofthought/blob/main/Finetuned_Qwen_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

use the gpu in colab and this should run fine

In [ ]:
!pip install unsloth -q
!pip install trl datasets transformers accelerate peft -q

In [ ]:
#encode the dataset and add final answer column for evaluation
def rot13(ans):
  if "\n" in ans:
    splits = ans.split("\n")
    return '\n'.join([rot13(a) for a in splits])
  caps = [a for a in range(65, 91)]
  lower = [a for a in range(97, 122)]
  out = []
  for i in ans:
    ordy = ord(i)
    if ordy in caps:
      if ordy > 77:
        out.append(chr(ordy-13))
      else:
        out.append(chr(ordy+13))
    elif ordy in lower:
      if ordy > 109:
        out.append(chr(ordy-13))
      else:
        out.append(chr(ordy+13))
    else:
      out.append(i)
  return ''.join(out)

def finalanswer(example):
  answer = example["answer"]
  # Find the index of '####'
  hash_index = answer.rfind("####")
  if hash_index != -1:
    # Extract the substring after '####' and strip whitespace
    return answer[hash_index + 4:].strip()
  else:
    # If '####' is not found, return the original answer or handle as an error
    return ""


from datasets import load_dataset
dataset = load_dataset("gsm8k", "main")
traindata = dataset["train"]  # 7,473 examples
testdata  = dataset["test"]   # 1,319 examples
def encode_answer(example):
    example["encoded_answer"] = rot13(example["answer"])
    return example
def final_answer(example):
    example["final_answer"] = finalanswer(example)
    return example

traindata = traindata.map(encode_answer)
traindata = traindata.map(final_answer)
testdata = testdata.map(final_answer)

In [ ]:
#format the dataset for good prompting & training
def format_example(example):
    return {
        "text": (
            f"Solve this maths problem. Think step by step inside <think> tags "
            f"encoded in ROT13, then give the final answer in plain English.\n\n"
            f"Question: {example['question']}\n"
            f"<think>\n{example['encoded_answer']}\n</think>\n"
            f"Answer: {example['final_answer']}"
        )
    }

train_formatted = traindata.map(format_example)

In [ ]:
#train the model on the dataset
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

train_subset = train_formatted.select(range(200))

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = "Qwen/Qwen2.5-3B-Instruct",
    max_seq_length = 1024,
    load_in_4bit   = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r              = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj"],
    lora_alpha     = 16,
    lora_dropout   = 0.05,
)

trainer = SFTTrainer(
    model              = model,
    tokenizer          = tokenizer,
    train_dataset      = train_subset, #replace with train_formatted for bigger
    dataset_text_field = "text",
    max_seq_length     = 1024,
    args               = TrainingArguments(
        output_dir                  = "./qwen_rot13",
        num_train_epochs            = 1, #higher means more train (3)
        per_device_train_batch_size = 4, #less batch size bigger
        gradient_accumulation_steps = 2,
        learning_rate               = 2e-6,
        lr_scheduler_type           = "cosine",
        warmup_ratio                = 0.1,
        fp16                        = True,
        logging_steps               = 10,
        optim                       = "adamw_8bit",
    ),
)

trainer.train()

In [ ]:
#Evaluate on the test dataset
FastLanguageModel.for_inference(model)

def evaluate(question, expected_answer):
    prompt = (
        f"Solve this maths problem. Think step by step inside <think> tags "
        f"encoded in ROT13, then give the final answer in plain English.\n\n"
        f"Question: {question}\n<think>\n"
    )
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    output_ids = model.generate(**inputs, max_new_tokens=512,
                                 temperature=0.0, do_sample=False)
    response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:],
                                skip_special_tokens=True)
    print(f"Expected: {expected_answer}")
    print(f"Got:      {response}")

evlauate(testdata['question'][0], testdata['final_answer'][0])